# Tratamento dos valores nulos (snull)

Parte dos 7 arquivos `dataset_*_perfil.csv` e `dataset_*_13cat.csv` e resolve os valores nulos restantes,
gravando o resultado como `dataset_*_snull.csv` na mesma pasta.

**Pré-requisito:** rodar `tratamento_datasets_13cat` antes.
Para a inspeção visual das distribuições, ver `boxplots_nulos`.

### Regras aplicadas a cada coluna numérica com nulos

| # | Condição | Ação |
|---|---|---|
| 1 | mais de 20% de nulos **e** faixa de temperatura > 35 °C | **excluir** a coluna |
| 2 | possui outlier extremo (fora de 2,5x o IQR) | preencher com a **mediana** |
| 3 | zero outliers, ou apenas moderados (dentro de 2,5x o IQR) | preencher com a **média** |

As regras são avaliadas nessa ordem: a exclusão vem primeiro, e uma coluna excluída
não é imputada.

### Duas decisões de implementação

**A faixa é medida sem os outliers** — é o intervalo entre o menor e o maior valor
dentro da cerca de 1,5x o IQR (os bigodes do boxplot), e não a amplitude bruta. Um
único ponto distante inflaria a amplitude e faria a coluna ser excluída por causa
dele, não por causa da dispersão real dos dados.

**A escolha entre média e mediana segue a assimetria.** A média é sensível a valores
extremos: preencher com ela uma coluna que tem pontos muito distantes puxa o valor
imputado na direção deles. A mediana não se move com isso. Onde não há extremo, a
média usa a informação de todos os pontos e é a estimativa mais estável.

As duas versões da base são tratadas na mesma execução. A adotada na entrega é a de perfil de perdas; a de 13 categorias permanece porque integra a grade de experimentos que compara as duas codificações da coluna de perdas.

## 1. Configuração e leitura

Os limiares ficam em constantes no topo, para que ajustar a regra não exija procurar
números soltos no meio do código.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import glob

import numpy as np
import pandas as pd

PCT_NULOS_MAX = 20.0     # acima disso a coluna e candidata a exclusao
FAIXA_MAX = 35.0         # em graus C, medida sem os outliers
MULT_EXTREMO = 2.5       # multiplo do IQR que define outlier extremo
SUFIXO_SAIDA = "_snull"

# A quarta sprint adota apenas a codificacao de 13 categorias, e e so ela que
# esta pasta processa. A versao de perfil de perdas permanece na pasta da terceira
# sprint, como registro daquela entrega, e nao e regenerada aqui.
#
# Consequencia medida da restricao, para que ela fique registrada: a formulacao
# que escolhe a codificacao por regiao alcanca 6,327 graus e escolhe perfil de
# perdas em cinco das sete regioes; restrita a 13 categorias, a mesma escolha
# passa a render -0,057, isto e, deixa de render. O ganho da quarta sprint vem do
# agrupamento do motivo, nao da liberdade de trocar de codificacao.
VERSOES = ("_13cat",)

# O nome da versao integra a chave de cada conjunto, de modo que os arquivos de
# saida nao colidam entre si: dataset_mainback_perfil_snull.csv e
# dataset_mainback_13cat_snull.csv.
arquivos = []
for versao in VERSOES:
    encontrados = sorted(a for a in glob.glob(f"dataset_*{versao}.csv")
                         if SUFIXO_SAIDA not in a)
    if not encontrados:
        print(f"AVISO: nenhum arquivo encontrado para a versao {versao}")
    arquivos.extend(encontrados)

print("Datasets encontrados:", len(arquivos))

datasets = {}
for arq in arquivos:
    nome = arq.replace("dataset_", "").replace(".csv", "")
    datasets[nome] = pd.read_csv(arq)
    print(f"  {nome:24s} -> {datasets[nome].shape}")

## 2. Estatísticas de cada coluna com nulos

`perfila_coluna` reúne num único dicionário tudo que as regras precisam: proporção de
nulos, IQR, faixa sem outliers e contagem de extremos. Calcular isso em um lugar só
evita repetir a conta dos quartis em cada regra.

In [ ]:
def perfila_coluna(serie):
    """Estatisticas de uma coluna numerica, ignorando os nulos."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1

    # cerca padrao de Tukey: separa o corpo dos dados dos pontos fora
    dentro = s[(s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)]
    faixa = dentro.max() - dentro.min() if len(dentro) else 0.0

    extremos = int(((s <= q1 - MULT_EXTREMO * iqr) |
                    (s >= q3 + MULT_EXTREMO * iqr)).sum())

    n_nulos = int(serie.isna().sum())
    return {
        "nulos": n_nulos,
        "pct_nulos": 100 * n_nulos / len(serie),
        "IQR": iqr,
        "faixa": faixa,
        "extremos": extremos,
        "media": s.mean(),
        "mediana": s.median(),
    }


def colunas_com_nulos(df):
    """Colunas numericas que possuem ao menos um valor nulo."""
    return [c for c in df.columns
            if df[c].isna().any() and pd.api.types.is_numeric_dtype(df[c])]

## 3. Decidir a ação de cada coluna

`decide_acao` traduz as três regras em uma string (`excluir`, `mediana` ou `media`).
Separar a decisão da execução deixa o plano inteiro visível antes de qualquer dado ser
alterado — dá para conferir a tabela e só então aplicar.

In [ ]:
def decide_acao(perfil):
    """Aplica as tres regras, na ordem, e devolve (acao, motivo)."""
    if perfil["pct_nulos"] > PCT_NULOS_MAX and perfil["faixa"] > FAIXA_MAX:
        return "excluir", (f"{perfil['pct_nulos']:.1f}% de nulos e faixa "
                           f"{perfil['faixa']:.2f} °C")
    if perfil["extremos"] > 0:
        return "mediana", (f"{perfil['extremos']} outlier(s) alem de "
                           f"{MULT_EXTREMO}x o IQR")
    return "media", "sem outliers extremos"


plano = []
for nome, df in datasets.items():
    for col in colunas_com_nulos(df):
        perfil = perfila_coluna(df[col])
        acao, motivo = decide_acao(perfil)
        plano.append({
            "dataset": nome,
            "coluna": col,
            "nulos": perfil["nulos"],
            "% nulos": round(perfil["pct_nulos"], 1),
            "IQR": round(perfil["IQR"], 2),
            "faixa": round(perfil["faixa"], 2),
            "extremos": perfil["extremos"],
            "acao": acao,
            "valor": (np.nan if acao == "excluir"
                      else round(perfil[acao], 2)),
            "motivo": motivo,
        })

plano = pd.DataFrame(plano)
print(plano["acao"].value_counts().to_string())
plano

## 4. Aplicar

O laço percorre o plano e altera uma cópia dos datasets. Como o plano já foi decidido
na célula anterior, aqui só resta executar — e reexecutar esta célula não muda o
resultado, porque a decisão não depende do estado já modificado.

In [ ]:
tratados = {nome: df.copy() for nome, df in datasets.items()}

for linha in plano.itertuples():
    df = tratados[linha.dataset]
    if linha.acao == "excluir":
        df.drop(columns=[linha.coluna], inplace=True)
        print(f"{linha.dataset:14} {linha.coluna:32} excluida  ({linha.motivo})")
    else:
        df[linha.coluna] = df[linha.coluna].fillna(linha.valor)
        print(f"{linha.dataset:14} {linha.coluna:32} "
              f"{linha.nulos} nulo(s) -> {linha.acao} {linha.valor}")

print("\nNulos restantes por dataset:")
for nome, df in tratados.items():
    restantes = int(df.isna().sum().sum())
    print(f"  {nome:16s} {df.shape}  nulos={restantes}")

## 5. Verificação

Antes de gravar, conferimos que nenhum nulo sobrou e que as colunas excluídas são
exatamente as previstas no plano. Se alguma dessas checagens falhar, o `assert`
interrompe o notebook em vez de deixar passar um arquivo inconsistente.

In [ ]:
esperado_excluir = set(
    zip(plano.loc[plano["acao"] == "excluir", "dataset"],
        plano.loc[plano["acao"] == "excluir", "coluna"])
)

for nome, df in tratados.items():
    assert df.isna().sum().sum() == 0, f"{nome} ainda tem nulos"
    assert len(df) == len(datasets[nome]), f"{nome} mudou de numero de linhas"

    perdidas = set(datasets[nome].columns) - set(df.columns)
    previstas = {c for (d, c) in esperado_excluir if d == nome}
    assert perdidas == previstas, f"{nome}: colunas removidas {perdidas} != {previstas}"

print("Verificacao OK -- nenhum nulo restante, linhas preservadas, "
      "apenas as colunas previstas foram removidas.")

## 6. Gravar os arquivos `_snull`

In [ ]:
for nome, df in tratados.items():
    destino = f"dataset_{nome}{SUFIXO_SAIDA}.csv"
    df.to_csv(destino, index=False)
    print(f"  {destino:40s} {df.shape}")

## 7. Resumo das mudanças

In [ ]:
resumo = pd.DataFrame([
    {
        "dataset": nome,
        "colunas antes": datasets[nome].shape[1],
        "colunas depois": df.shape[1],
        "colunas excluidas": datasets[nome].shape[1] - df.shape[1],
        "nulos antes": int(datasets[nome].isna().sum().sum()),
        "nulos depois": int(df.isna().sum().sum()),
    }
    for nome, df in tratados.items()
])

print(f"{len(plano)} colunas tratadas  |  "
      f"{(plano['acao'] == 'excluir').sum()} excluidas, "
      f"{(plano['acao'] == 'mediana').sum()} por mediana, "
      f"{(plano['acao'] == 'media').sum()} por media")
resumo

## Conferência entre as duas versões

As duas versões diferem apenas na codificação da coluna de perdas, de modo que as colunas-alvo devem ser idênticas entre elas. Sendo idênticas, o plano de tratamento decidido para cada componente também deve coincidir.

A conferência abaixo verifica essa correspondência. Uma divergência indicaria que as versões não foram geradas a partir da mesma base, o que invalidaria a comparação entre elas.

In [ ]:
pares = {}
for chave in datasets:
    for versao in VERSOES:
        if chave.endswith(versao):
            pares.setdefault(chave[: -len(versao)], {})[versao] = chave

divergencias = []
for area, mapa in sorted(pares.items()):
    if len(mapa) < 2:
        divergencias.append(f"{area}: presente em apenas uma versao")
        continue
    a = plano[plano["dataset"] == mapa[VERSOES[0]]][["coluna", "acao"]]
    b = plano[plano["dataset"] == mapa[VERSOES[1]]][["coluna", "acao"]]
    juncao = a.merge(b, on="coluna", how="outer", suffixes=("_a", "_b"), indicator=True)
    so_um = juncao[juncao["_merge"] != "both"]
    acao_dif = juncao[(juncao["_merge"] == "both") & (juncao["acao_a"] != juncao["acao_b"])]
    if len(so_um) or len(acao_dif):
        divergencias.append(f"{area}: {len(so_um)} coluna(s) em apenas uma versao, "
                            f"{len(acao_dif)} com acao diferente")

print(f"Areas conferidas: {len(pares)}")
if divergencias:
    print("DIVERGENCIAS encontradas:")
    for d in divergencias:
        print("  ", d)
else:
    print("As duas versoes recebem o mesmo plano de tratamento em todas as areas,")
    print("o que confirma que possuem colunas-alvo identicas.")

# Com uma unica versao processada, a conferencia cruzada nao tem objeto: ela
# existe para garantir que as duas versoes recebam o mesmo plano, e nao ha duas.
# A verificacao permanece no codigo e volta a valer se VERSOES voltar a ter duas
# entradas; o que se faz aqui e declarar a sua inaplicabilidade, e nao ignora-la.
if len(VERSOES) < 2:
    print()
    print("Conferencia cruzada entre versoes: NAO SE APLICA nesta sprint, que processa")
    print(f"apenas {VERSOES[0]}. A conferencia volta a valer se a outra versao for reincluida.")
else:
    assert not divergencias, "as versoes divergem no plano de tratamento de nulos"